# Crop Yield Capstone – Team Notebook (11 Members, 11 Sections)
**Machine Learning and AI (Essentials) · Thrive Campus · Group project**  |  Crop: **Rice, paddy** (change `CROP` in Section 2 if your group picks another)

You are the data science team for a national agriculture ministry. The ministry asks three questions: **How much will we harvest?** (regression) · **Will this be a bad year?** (classification) · **What kinds of growing years do we have?** (clustering).

This notebook splits the whole capstone into **11 sections, one per team member**. Each section has `TODO` and `HINT` comments telling you what to write. A finished solution exists separately (`Crop_Yield_Capstone_SOLUTION.ipynb`) for the teaching team – don't ask for it; learn by doing.

## Team roster (fill in)
| Section | Member | Task | Marks area | GitHub username |
|---|---|---|---|---|
| 1 | Member 1 | Set up and load data | Part 1 | |
| 2 | Martha | Filter crop + 3 observations | Part 1 | |
| 3 | Martha | Clean the data + cleaning log | Cleaning (15) | |
| 4 | Martha | Charts 1 and 2 | Charts (15) | |
| 5 | Martha | Charts 3 and 4 | Charts (15) | |
| 6 | Martha | Regression: features + split | Regression (20) | |
| 7 | Joseph | Regression: two models, scores, coefficients | Regression (20) | |
| 8 | Joseph | Classification: target, leakage, split | Classification (20) | |
| 9 | Yasir | Classification: models, metrics, confusion matrix | Classification (20) | |
| 10 | Yasir | Clustering | Clustering (10) | |
| 11 | Yasir | Grid Search tuning + final summary | Tuning (5) | |

Report (15 marks) and presentation: **everyone** contributes – each member writes up their own section for the report and presents it.

## How the team works together
1. **Sections run top to bottom and depend on each other.** Each section lists *Receives* (variables you can use from earlier sections) and *Hands over* (variables the next person needs). Keep those variable names exactly.
2. **Work on your own branch**, e.g. `section-07-regression`, and open a pull request. Everyone commits – no commits means no group mark.
3. **Weekly checkpoints** (see the milestone table in the brief): Members 1–2 → Milestones 1–2; 3 → M3; 4–5 → M4; 6–7 → M5; 8–9 → M6; 10–11 → M7; everyone → M8 (report and slides). Missing two checkpoints in a row is how groups fail.
4. **Until the person before you has pushed**, use the fall-back lines in each section to test your code.
5. Before the final hand-in, one person runs **Kernel → Restart & Run All**. An error at any point costs −5.

## Rules that run through everything
* **Every model result ends with a sentence starting “This means the ministry should…”.** Technical work without a recommendation scores zero for that section.
* **Honest numbers only.** A poor result explained well beats an inflated one. R² above 0.95 → probably scored on training data. Accuracy of 1.00 → leakage.
* **Comment every code line you write** so a teammate can explain it. AI assistants are allowed for explanations – note where you used one in the table at the very bottom.
* Penalties: notebook errors −5 · regression scored on training data −7 · yield column in classification inputs −10.

---
# SECTION 1 · Part 1 – Set up and load the data
**Assigned to:** Member 1 (write your name here: ________)  |  **Marks area:** Part 1 (set-up, milestone 1)

**Receives:** nothing (you are first)
**Hands over:** `raw` (the full DataFrame), `RANDOM_STATE`, `save_fig()`, `RESULTS`, all imports

**Goal:** make the notebook run on anyone's computer and load `yield_df.csv`.

**Step 1.1 – Imports and settings.**

In [ ]:
# TODO: import every library the notebook needs, in ONE cell (other members will add to this list if they need more).
# HINT: numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns,
#       from sklearn: model_selection (train_test_split, GridSearchCV), preprocessing (StandardScaler), pipeline (Pipeline),
#       linear_model (LinearRegression, LogisticRegression), ensemble (RandomForestRegressor, RandomForestClassifier),
#       dummy (DummyClassifier), cluster (KMeans), metrics (mean_squared_error, r2_score, accuracy_score, precision_score,
#       recall_score, confusion_matrix, ConfusionMatrixDisplay, silhouette_score)

# TODO: set RANDOM_STATE = 42 so everyone gets identical numbers.
# TODO: create the folders 'figures' and 'results' (HINT: os.makedirs(..., exist_ok=True)).
# TODO: create an empty dictionary RESULTS = {} - teammates will store headline numbers in it.
# TODO: write a small function save_fig(fig, name) that saves a chart to figures/<name>.png (HINT: fig.savefig(..., dpi=160, bbox_inches='tight')).
# YOUR CODE HERE
# Core libraries
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

# Modeling
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, confusion_matrix, classification_report
)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", None)
np.random.seed(42)


: 

**Step 1.2 – Load the file.**

In [ ]:
# TODO: read yield_df.csv into a DataFrame called `raw`.
# HINT: pd.read_csv("yield_df.csv").  FileNotFoundError? The CSV must be in the SAME folder as this notebook.
# TODO: print raw.shape, show raw.head() and raw.info().
# YOUR CODE HERE
# TODO: change this to your assigned crop (must match spelling in 'Item' exactly)
CROP = "Maize"

try:
    raw = pd.read_csv("notebook/yield_df.csv")
except FileNotFoundError:
    raise FileNotFoundError(
        "yield_df.csv not found in this folder. "
        "Download it from kaggle.com/datasets/patelris/crop-yield-prediction-dataset "
        "and place it next to this notebook."
    )

if 'Unnamed: 0' in raw.columns:
    raw = raw.drop(columns=['Unnamed: 0'])

print(f"Full dataset shape: {raw.shape}")
print(f"Number of crops (Item): {raw['Item'].nunique()}")
raw['Item'].unique()


**Step 1.3 – Write 2–3 sentences:** what does one row represent? Which column is the leftover index? How many crops and countries are there?

In [ ]:
# HINT: raw['Item'].nunique(), raw['Area'].nunique()
# YOUR CODE HERE
# Filter to your crop
df = raw[raw['Item'] == CROP].copy()

if df.shape[0] == 0:
    close_matches = [c for c in raw['Item'].unique() if CROP.lower() in str(c).lower() or str(c).lower() in CROP.lower()]
    raise ValueError(
        f"Your filter returned 0 rows for CROP='{CROP}'. "
        f"This is almost always spelling/capitalisation (watch for commas like 'Rice, paddy'). "
        f"Possible close matches in the data: {close_matches}"
    )

print(f"'{CROP}' rows: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Countries (Area) represented: {df['Area'].nunique()}")
print(f"Year range: {df['Year'].min()}-{df['Year'].max()}")
df.head()


---
# SECTION 2 · Part 1 (continued) – Filter to our crop and write three observations
**Assigned to:** Member 2 (write your name here: ________)  |  **Marks area:** Part 1 (milestones 1–2): three observations written

**Receives:** `raw`
**Hands over:** `df` (only your crop's rows, still un-cleaned)

**Goal:** narrow the data to one staple crop and notice things *before* cleaning.

**Step 2.1 – Look at the crop names.**

In [ ]:
# TODO: print how many rows each crop has.
# HINT: raw["Item"].value_counts().  Names must match EXACTLY: "Rice, paddy" has a comma; "Rice" alone matches nothing.
# YOUR CODE HERE
print(raw["Item"].value_counts())


**Step 2.2 – Filter.**

In [ ]:
# TODO: set CROP = "Rice, paddy" (or your group's crop) and create df = raw[raw["Item"] == CROP].copy()
# TODO: add an assert that len(df) > 0 so a spelling mistake fails loudly.
# TODO: print the number of rows, countries (df['Area'].nunique()) and the first/last year.
# YOUR CODE HERE
CROP = "Potatoes"                              # our crop (note the comma)
df = raw[raw["Item"] == CROP].copy()              # keep only this crop's rows
assert len(df) > 0, f"No rows for CROP='{CROP}' - check spelling, capitals and the comma"   # fail loudly on typos
print("Rows:", len(df))                           # how many rows
print("Countries:", df["Area"].nunique())         # how many countries
print("Years:", df["Year"].min(), "to", df["Year"].max())   # first and last year

**Step 2.3 – Gather evidence for your observations.**

In [ ]:
# TODO: print df.describe() for the four numeric columns.
# TODO: count how many rows each (Area, Year) pair has.  HINT: df.groupby(["Area","Year"]).size().value_counts()
#       If it is not all 1s, the data is NOT one-row-per-country-per-year - Member 3 needs to know!
# YOUR CODE HERE
num_cols_raw = ["hg/ha_yield", "average_rain_fall_mm_per_year", "pesticides_tonnes", "avg_temp"]   # the 4 numeric columns
df_num = df.copy()                                # numeric copy used only for evidence (cleaning comes in Section 3)
for c in num_cols_raw:                            # loop over the numeric columns
    df_num[c] = pd.to_numeric(df_num[c], errors="coerce")   # text -> number (bad entries -> NaN)
display(df_num[num_cols_raw].describe())          # summary statistics
per_pair = df.groupby(["Area", "Year"]).size()    # rows for each country-year
print("How many rows does each (Area, Year) pair have?")      # explain the table below
print(per_pair.value_counts().sort_index())       # value -> number of pairs
if (per_pair > 1).any():                          # more than one row for some country-years?
    print("NOT one-row-per-country-per-year: Member 3 must collapse the repeats.")
else:
    print("Exactly one row per country-year.")

**Step 2.4 – Three observations.** Replace the numbered lines below with your own, each containing a specific number.

In [ ]:
# (No code needed - write in the markdown cell that follows.)
# Observations are written from the numbers, so they update with whatever file is loaded.
n_pairs = len(per_pair)                                       # distinct country-years
n_repeat = int((per_pair > 1).sum())                          # country-years appearing more than once
max_rep = int(per_pair.max())                                 # worst repetition
yld_all = df_num["hg/ha_yield"]                               # yield column (numeric)
ctry_mean = df_num.groupby("Area")["hg/ha_yield"].mean()      # mean yield per country
between_share = df_num.groupby("Area")["hg/ha_yield"].transform("mean").var() / yld_all.var()   # variance explained by country alone
rain_vals_max = int(df_num.groupby("Area")["average_rain_fall_mm_per_year"].nunique().max())    # rain values per country
pest = df_num["pesticides_tonnes"]                            # pesticide column (numeric)
rain_note = ("so it describes a country's typical climate, not each year's weather"
             if rain_vals_max == 1 else "so it varies only a little within a country")
say("**Observations (written from the data):**\n\n"
    f"1. **Repeated rows.** {len(df):,} rows but only {n_pairs:,} distinct country-years: {n_repeat:,} of them "
    f"({n_repeat/n_pairs:.0%}) appear more than once (up to {max_rep} rows), so rows are not independent observations. "
    f"The data covers {df['Area'].nunique()} countries from {df['Year'].min()} to {df['Year'].max()}.\n\n"
    f"2. **Country dominates yield.** Median yield is {yld_all.median():,.0f} hg/ha (about {yld_all.median()/1e4:.1f} t/ha), "
    f"but country averages run from {ctry_mean.min():,.0f} ({ctry_mean.idxmin()}) to {ctry_mean.max():,.0f} ({ctry_mean.idxmax()}), "
    f"a {ctry_mean.max()/ctry_mean.min():.1f}-fold gap, and country alone accounts for about {between_share:.0%} of all variation in yield.\n\n"
    f"3. **Inputs are skewed and rainfall is not yearly.** Pesticide use has a median of {pest.median():,.0f} t but a maximum of "
    f"{pest.max():,.0f} t ({pest.max()/max(pest.median(),1):,.0f}x the median) with {int((pest==0).sum())} zero rows; "
    f"rainfall has at most {rain_vals_max} distinct value(s) per country, {rain_note}.")

**Observations (edit me):**
1. TODO
2. TODO
3. TODO

---
# SECTION 3 · Part 2 – Clean the data (and fill the cleaning log)
**Assigned to:** Member 3 (write your name here: ________)  |  **Marks area:** Cleaning (15 marks) – milestone 3

**Receives:** `df`
**Hands over:** `rice` (clean, ONE row per country-year, columns renamed: yield_hg_ha, rain_mm, pesticides_t, temp_c), `cleaning_df` (the log)

**Goal:** every cleaning decision has a written reason: types, duplicates, missing values, outliers.

**Step 3.1 – Start a cleaning log.**

In [ ]:
# TODO: create an empty list cleaning_log and a helper log(step, found, decision, reason) that appends a dict to it.
# HINT: cleaning_log.append({"Step": step, "What we found": found, "What we did": decision, "Why": reason})
# YOUR CODE HERE
# TODO: create an empty list cleaning_log and a helper log(step, found, decision, reason) that appends a dict to it.
# HINT: cleaning_log.append({"Step": step, "What we found": found, "What we did": decision, "Why": reason})
# ---- solution ----
cleaning_log = []                                 # one dict per cleaning decision

def log(step, found, decision, reason):
    """Record one cleaning decision."""
    cleaning_log.append({"Step": step, "What we found": found, "What we did": decision, "Why": reason})   # add a row to the log

**Step 3.2 – Index column and names.**

In [ ]:
# TODO: drop the 'Unnamed: 0' column and rename the long columns:
#   hg/ha_yield -> yield_hg_ha, average_rain_fall_mm_per_year -> rain_mm, pesticides_tonnes -> pesticides_t, avg_temp -> temp_c
# HINT: df.drop(columns=[...]) and df.rename(columns={...}).   Then call log(...) to record what you did and WHY.
# ---- solution ----
# YOUR CODE HERE
n_rows_start = len(df)                            # rows at the start of cleaning
df = df.drop(columns=["Unnamed: 0"], errors="ignore")   # drop the leftover index column if present
df = df.rename(columns={"hg/ha_yield": "yield_hg_ha",          # short, code-friendly names
                        "average_rain_fall_mm_per_year": "rain_mm",
                        "pesticides_tonnes": "pesticides_t",
                        "avg_temp": "temp_c"})
log("Index column and names",
    "A leftover 'Unnamed: 0' index column (if present) and four long column names",
    "Dropped the index column and renamed the four numeric columns",
    "The index repeats row numbers (no information, and could be mistaken for a feature); short names avoid typos in later code")
print(df.columns.tolist())                        # confirm the new names

**Step 3.3 – Data types.**

In [ ]:
# TODO: print df.dtypes. Convert every numeric column with pd.to_numeric(df[c], errors="coerce").
# HINT: "could not convert string to float" means a number is stored as text.  Record the result in the log.
# YOUR CODE HERE
print(df.dtypes)                                  # which columns are stored as text?
num_cols = ["yield_hg_ha", "rain_mm", "pesticides_t", "temp_c", "Year"]   # columns that must be numbers
text_cols = [c for c in num_cols if not pd.api.types.is_numeric_dtype(df[c])]   # stored as text before conversion
unparsable = {}                                   # new NaNs created per column
for c in num_cols:                                # convert each one
    before = df[c].isna().sum()                   # NaNs before conversion
    df[c] = pd.to_numeric(df[c], errors="coerce") # text -> number, junk -> NaN
    unparsable[c] = int(df[c].isna().sum() - before)   # NaNs created by the conversion
print("Values that could not be read as numbers:", unparsable)
log("Data types",
    f"Columns stored as text: {text_cols if text_cols else 'none'}; unreadable entries per column: {unparsable}",
    "Converted every numeric column with pd.to_numeric(errors='coerce')",
    "Models need numbers; entries that are not numbers (such as '..') cannot be used, so they become missing and are handled in the next step")

**Step 3.4 – Missing values.**

In [ ]:
# TODO: count missing values per column (df.isna().sum()), then decide: drop rows, fill, or nothing needed?  Log the decision.
# HINT: "Input contains NaN" later in the project means this step was incomplete.
# YOUR CODE HERE
print(df.isna().sum())                            # missing values per column
n_before_na = len(df)                             # rows before dropping
df = df.dropna(subset=["Area", "Year"] + ["yield_hg_ha", "rain_mm", "pesticides_t", "temp_c"])   # drop rows missing any needed value
df["Year"] = df["Year"].astype(int)               # years are whole numbers
n_missing_dropped = n_before_na - len(df)         # how many rows went
print("Rows dropped for missing values:", n_missing_dropped)
log("Missing values",
    f"{n_missing_dropped} rows had a missing yield, rainfall, pesticide, temperature, country or year value",
    "Dropped those rows" if n_missing_dropped else "Nothing to do",
    "Filling in a yield, rainfall or temperature would invent data about a real country-year; the number of rows lost is small compared with the file")

**Step 3.5 – Exact duplicates.**

In [ ]:
# TODO: count and remove rows that are identical in every column.  HINT: df.duplicated().sum(), df.drop_duplicates().  Log it.
# YOUR CODE HERE
n_exact = int(df.duplicated().sum())              # rows identical in every column
df = df.drop_duplicates()                         # keep one copy of each
print("Exact duplicate rows removed:", n_exact)
log("Exact duplicates",
    f"{n_exact} rows identical in every column",
    "Removed the extra copies",
    "A repeated identical row adds no information and would count the same observation twice in every model and score")

**Step 3.6 – The repeated country-years (important!).**

In [ ]:
# TODO: for each (Area, Year), count distinct values per column.  HINT: df.groupby(["Area","Year"])[cols].nunique()
#       Which column(s) disagree between repeated rows?
# TODO: decide how to get ONE row per country-year (HINT: groupby(["Area","Year"], as_index=False).agg(...), averaging the column that disagrees
#       and taking the first value of the rest).  Save the result as `rice`.
# TODO: prove it worked with assert not rice.duplicated(["Area","Year"]).any()
# TODO: log your decision, including how many rows were removed and WHY (think: what would happen to a train/test split otherwise?)
# YOUR CODE HERE
cols = ["yield_hg_ha", "rain_mm", "pesticides_t", "temp_c"]                 # columns to compare within a country-year
nun = df.groupby(["Area", "Year"])[cols].nunique()                           # distinct values per column per country-year
disagree = (nun > 1).sum()                                                   # how many country-years disagree, per column
print("Country-years where repeated rows DISAGREE, by column:")
print(disagree)
disagree_cols = [c for c in cols if (nun[c] > 1).any()]                      # columns that differ between repeats
agg_map = {"Item": "first"}                                                  # crop name is the same, keep the first
agg_map.update({c: ("mean" if c in disagree_cols else "first") for c in cols})   # average the disagreeing columns, keep the rest
n_before_collapse = len(df)                                                  # rows before collapsing
rice = df.groupby(["Area", "Year"], as_index=False).agg(agg_map)             # ONE row per country-year
rice = rice[["Area", "Item", "Year"] + cols]                                 # tidy column order
assert not rice.duplicated(["Area", "Year"]).any()                           # prove it worked
removed_pairs = n_before_collapse - len(rice)                                # rows removed by collapsing
print(f"Collapsed {n_before_collapse:,} rows into {len(rice):,} country-years (removed {removed_pairs:,}); averaged: {disagree_cols}")
log("Repeated country-years",
    f"{removed_pairs} extra rows shared a country and year with another row; the columns that disagreed were {disagree_cols if disagree_cols else 'none'}",
    f"Collapsed to one row per country-year, averaging {disagree_cols if disagree_cols else 'nothing'} and keeping the first value of the others",
    "Repeats of one country-year are not independent: a random split would put near-identical rows in both train and test, inflating the test score, and some years would count more than others")

**Step 3.7 – Ranges and outliers.**

In [ ]:
# TODO: print rice.describe(). Check yields > 0, rainfall > 0, pesticides >= 0 (use assert).
# TODO: count outliers in yield with the IQR rule (below Q1-1.5*IQR or above Q3+1.5*IQR). Decide keep or remove, and log WHY.
# TODO: check: does rain_mm take more than one value inside a country?  HINT: rice.groupby("Area")["rain_mm"].nunique().max()
#       (Members 6 and 8 need the answer!)
# YOUR CODE HERE
display(rice.describe())                                                     # ranges of every column
assert (rice["yield_hg_ha"] > 0).all(), "A yield is zero or negative"        # yields must be positive
assert (rice["rain_mm"] > 0).all(), "A rainfall value is zero or negative"   # rainfall must be positive
assert (rice["pesticides_t"] >= 0).all(), "A pesticide value is negative"    # pesticides cannot be negative
log("Ranges", "Yield and rainfall are all > 0 and pesticides all >= 0", "Nothing removed", "Values are physically possible, so there is no reason to delete rows")
q1, q3 = rice["yield_hg_ha"].quantile([0.25, 0.75])                          # quartiles of yield
iqr = q3 - q1                                                                # inter-quartile range
low_fence, high_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr                       # IQR rule limits
outliers = rice[(rice["yield_hg_ha"] < low_fence) | (rice["yield_hg_ha"] > high_fence)]   # rows outside the fences
n_out = len(outliers)                                                        # how many
out_countries = outliers["Area"].nunique()                                   # from how many countries
share_in_country = (outliers.groupby("Area").size() / rice.groupby("Area").size()).dropna()   # fraction of each country's years that are outliers
print(f"IQR outliers: {n_out} rows ({n_out/len(rice):.1%}) from {out_countries} countries; fences {low_fence:,.0f} to {high_fence:,.0f}")
print("In the countries involved, the median share of their years that are outliers:",
      f"{share_in_country.median():.0%}" if len(share_in_country) else "n/a")
log("Outliers in yield (IQR rule)",
    f"{n_out} rows ({n_out/len(rice):.1%}) outside {low_fence:,.0f}-{high_fence:,.0f} hg/ha, from {out_countries} countries",
    "Kept them",
    "They are concentrated in the same countries year after year (real differences between farming systems, not typing errors); deleting them would remove the ministry's best and worst cases")
rain_per_country_max = int(rice.groupby("Area")["rain_mm"].nunique().max())  # distinct rainfall values inside one country
print("Largest number of different rain_mm values inside any one country:", rain_per_country_max)
RESULTS["rain_values_per_country_max"] = rain_per_country_max               # Members 6 and 8 need this

**Step 3.8 – Show the log and save the clean data.**

In [ ]:
# TODO: turn cleaning_log into a DataFrame `cleaning_df`, display it, and save results/cleaning_log.csv and results/rice_clean.csv
# TODO: print how many rows you started with and ended with.
# YOUR CODE HERE
cleaning_df = pd.DataFrame(cleaning_log)                                     # log as a table
pd.set_option("display.max_colwidth", None)                                  # show the full text of each cell
display(cleaning_df)
cleaning_df.to_csv("results/cleaning_log.csv", index=False)                  # save the log
rice.to_csv("results/rice_clean.csv", index=False)                           # save the clean data
print(f"Started with {n_rows_start:,} rows, ended with {len(rice):,} clean country-year rows.")
RESULTS["rows_start"], RESULTS["rows_clean"] = n_rows_start, len(rice)       # headline numbers

### ✍️ Your written answer for Section 3 (double-click to edit)
> **This means the ministry should…** *(TODO: finish this sentence in plain English, using the numbers you found above. Maximum 3 sentences.)*

---
# SECTION 4 · Part 3 – Explore with charts (charts 1 and 2)
**Assigned to:** Member 4 (write your name here: ________)  |  **Marks area:** Charts (15 marks) – milestone 4 (shared with Member 5)

**Receives:** `rice`, `save_fig`
**Hands over:** charts saved in `figures/`

**Goal:** two charts, each with a title, labelled axes and a *specific* written interpretation (mention numbers; say what it means).

**Chart 1 – distribution of yield.**

In [ ]:
# TODO: histogram of rice["yield_hg_ha"] with a vertical line at the median.
# HINT: ax.hist(..., bins=40); ax.axvline(median, linestyle="--")
# TODO: title, x label WITH UNITS, y label, legend.  Save with save_fig(fig, "chart1_yield_distribution").
# YOUR CODE HERE
yld = rice["yield_hg_ha"]                                  # yield column
med = yld.median()                                         # median yield
fig, ax = plt.subplots(figsize=(8, 5))                     # new figure
ax.hist(yld, bins=40, color="#4C9F70", edgecolor="white")  # histogram
ax.axvline(med, linestyle="--", color="black", label=f"Median = {med:,.0f} hg/ha")   # median line
ax.set_title(f"Distribution of {CROP} yield (one value per country-year)")           # title
ax.set_xlabel("Yield (hg/ha)")                             # x label with units
ax.set_ylabel("Number of country-years")                   # y label
ax.legend()                                                # show legend
save_fig(fig, "chart1_yield_distribution")                 # save image
plt.show()                                                 # display

In [ ]:
skew = yld.skew()                                          # shape measure
p10, p90 = yld.quantile([0.1, 0.9])                        # middle 80% of country-years
shape = ("right-skewed (a long tail of very high-yield country-years)" if skew > 0.5
         else "left-skewed (a tail of very low-yield country-years)" if skew < -0.5 else "roughly symmetric")
say(f"**Interpretation of chart 1.** The distribution is {shape} (skewness {skew:.2f}). The median is {med:,.0f} hg/ha "
    f"(about {med/1e4:.1f} t/ha) against a mean of {yld.mean():,.0f}; the middle 80% of country-years run from {p10:,.0f} to {p90:,.0f} hg/ha, "
    f"a {p90/p10:.1f}-fold spread. For the ministry, a single national yield figure is a poor guide: a typical year in a high-yield "
    f"country is far above a good year in a low-yield one.")

**Interpretation of chart 1** (edit): What shape is it? What is the median? What does the spread imply for the ministry?

> TODO

**Chart 2 – yield over time.**

In [ ]:
# TODO: compute the mean yield per Year and draw a line chart. HINT: rice.groupby("Year")["yield_hg_ha"].mean()
# TODO: fit a straight line with np.polyfit(years, means, 1) and print the gain in hg/ha per year.
# TODO: title, axis labels, save_fig(fig, "chart2_yield_trend").
# YOUR CODE HERE
yearly = rice.groupby("Year")["yield_hg_ha"].mean()                 # mean yield each year
years = yearly.index.to_numpy(dtype=float)                          # years as numbers
slope, intercept = np.polyfit(years, yearly.to_numpy(), 1)          # straight-line trend
print(f"Trend: {slope:+,.0f} hg/ha per year")                       # gain per year
fig, ax = plt.subplots(figsize=(9, 5))                              # new figure
ax.plot(yearly.index, yearly.values, marker="o", label="Mean yield across countries")   # yearly means
ax.plot(yearly.index, slope * years + intercept, "--", color="red", label=f"Trend: {slope:+,.0f} hg/ha per year")   # trend line
ax.set_title(f"Mean {CROP} yield by year")                          # title
ax.set_xlabel("Year")                                               # x label
ax.set_ylabel("Mean yield (hg/ha)")                                 # y label
ax.legend()                                                         # legend
save_fig(fig, "chart2_yield_trend")                                 # save image
plt.show()                                                          # display

In [ ]:
trend_pct = slope / yearly.mean() * 100                             # trend as % of average yield per year
chg = yearly.diff()                                                 # year-on-year change
worst_year, worst_drop = int(chg.idxmin()), float(chg.min())        # biggest one-year fall
if abs(trend_pct) < 0.3:
    trend_txt = f"treat average yield as essentially flat (trend {slope:+,.0f} hg/ha per year)"
else:
    trend_txt = f"plan on a {'rising' if slope > 0 else 'falling'} baseline of about {slope:+,.0f} hg/ha per year ({trend_pct:+.1f}% of average yield)"
say("### Written answer for Section 4\n\n"
    f"> **This means the ministry should** {trend_txt}, but keep a safety margin for swings such as the {worst_drop:,.0f} hg/ha change in "
    f"{worst_year}, rather than assume each harvest will beat the last.")

### ✍️ Your written answer for Section 4 (double-click to edit)
> **This means the ministry should…** *(TODO: finish this sentence in plain English, using the numbers you found above. Maximum 3 sentences.)*

**Interpretation of chart 2 (edit):** Is the trend up or down? By how much? Why are there bumps?

> TODO

In [ ]:
n_by_year = rice.groupby("Year")["Area"].nunique()                  # countries reporting each year
best_year, best_gain = int(chg.idxmax()), float(chg.max())          # biggest one-year rise
say(f"**Interpretation of chart 2.** The trend is {'upward' if slope > 0 else 'downward'}: about {slope:+,.0f} hg/ha per year "
    f"({trend_pct:+.1f}% of the average yield), or roughly {slope*(years.max()-years.min()):+,.0f} hg/ha over {int(years.min())}-{int(years.max())}. "
    f"The bumps are large: the biggest fall was {worst_drop:,.0f} hg/ha in {worst_year} and the biggest rise {best_gain:+,.0f} in {best_year}. "
    f"Some of this is weather and policy, but the number of reporting countries also changes ({n_by_year.min()} to {n_by_year.max()} per year), "
    f"so part of every bump is just a different mix of countries.")

---
# SECTION 5 · Part 3 (continued) – Explore with charts (charts 3 and 4)
**Assigned to:** Member 5 (write your name here: ________)  |  **Marks area:** Charts (15 marks) – milestone 4

**Receives:** `rice`, `save_fig`
**Hands over:** charts saved in `figures/`; a correlation table

**Goal:** two more charts with the same standard as Section 4. Between you and Member 4 you must cover four DIFFERENT charts.

**Chart 3 – temperature vs yield.**

In [ ]:
# TODO: scatter plot of temp_c (x) vs yield_hg_ha (y) with a trend line. HINT: alpha=0.3 makes overlapping points readable.
# TODO: print the correlation: rice["temp_c"].corr(rice["yield_hg_ha"])
# TODO: title, axis labels with units, save_fig(fig, "chart3_temp_vs_yield")
# YOUR CODE HERE
r_temp = rice["temp_c"].corr(rice["yield_hg_ha"])                   # overall correlation
print(f"Correlation temp vs yield: {r_temp:.3f}")                   # print it
fig, ax = plt.subplots(figsize=(8, 5))                              # new figure
ax.scatter(rice["temp_c"], rice["yield_hg_ha"], alpha=0.3, s=18, color="#D1603D")   # points
m, b = np.polyfit(rice["temp_c"], rice["yield_hg_ha"], 1)           # trend line
xs = np.linspace(rice["temp_c"].min(), rice["temp_c"].max(), 100)   # x range
ax.plot(xs, m * xs + b, color="black", label=f"Trend: {m:+,.0f} hg/ha per degree C")   # draw it
ax.set_title(f"Temperature vs {CROP} yield")                        # title
ax.set_xlabel("Average temperature (degrees C)")                    # x label with units
ax.set_ylabel("Yield (hg/ha)")                                      # y label
ax.legend()                                                         # legend
save_fig(fig, "chart3_temp_vs_yield")                               # save image
plt.show()                                                          # display

In [ ]:
t_mean = rice.groupby("Area")["temp_c"].transform("mean")           # each country's average temperature
between_t = t_mean.var() / rice["temp_c"].var()                     # share of temperature variation that is between countries
t_dev = rice["temp_c"] - t_mean                                     # temperature relative to own country
y_dev = rice["yield_hg_ha"] - rice.groupby("Area")["yield_hg_ha"].transform("mean")   # yield relative to own country
within_t = t_dev.corr(y_dev)                                        # within-country correlation
strength = "strong" if abs(r_temp) > 0.6 else "moderate" if abs(r_temp) > 0.3 else "weak"
say(f"**Interpretation of chart 3.** Across all country-years, hotter goes with {'higher' if r_temp > 0 else 'lower'} yield, a {strength} link "
    f"(r = {r_temp:.2f}). **Warning:** {between_t:.0%} of the temperature variation is between countries, so the chart mostly compares hot-climate "
    f"countries with cool-climate ones rather than showing what happens when one country has a hotter year. Inside countries the link is "
    f"r = {within_t:.2f}, which is the better guide to the effect of a warm year, and neither number proves that temperature causes the difference.")

**Interpretation of chart 3:** TODO (direction, strength, and one WARNING about what the chart does NOT prove - hint: is the temperature difference mostly between countries or within them?)

**Chart 4 – pesticides vs yield.**

In [ ]:
# TODO: scatter plot of pesticides_t vs yield_hg_ha. Pesticide tonnes range from 0 to hundreds of thousands, so try a log x-axis
# HINT: ax.set_xscale("log") and plot rice["pesticides_t"] + 1 (so zeros can be drawn)
# TODO: print a rank correlation: rice["pesticides_t"].corr(rice["yield_hg_ha"], method="spearman")
# TODO: title, labels, save_fig(fig, "chart4_pesticides_vs_yield")
# YOUR CODE HERE
rho_pest = rice["pesticides_t"].corr(rice["yield_hg_ha"], method="spearman")   # rank correlation
print(f"Spearman correlation pesticides vs yield: {rho_pest:.3f}")      # print it
fig, ax = plt.subplots(figsize=(8, 5))                                  # new figure
ax.scatter(rice["pesticides_t"] + 1, rice["yield_hg_ha"], alpha=0.3, s=18, color="#3F6FB5")   # +1 so zeros can be drawn
ax.set_xscale("log")                                                    # log axis because values span orders of magnitude
ax.set_title(f"Pesticide use vs {CROP} yield")                          # title
ax.set_xlabel("Pesticide use + 1 (tonnes, log scale)")                  # x label with units
ax.set_ylabel("Yield (hg/ha)")                                          # y label
save_fig(fig, "chart4_pesticides_vs_yield")                             # save image
plt.show()                                                              # display

In [ ]:
lp = np.log1p(rice["pesticides_t"])                                     # log scale of pesticide use
lp_dev = lp - lp.groupby(rice["Area"]).transform("mean")                # relative to own country
within_p = lp_dev.corr(y_dev)                                           # within-country correlation
strength_p = "strong" if abs(rho_pest) > 0.6 else "moderate" if abs(rho_pest) > 0.3 else "weak"
say(f"**Interpretation of chart 4.** More pesticide goes with {'higher' if rho_pest > 0 else 'lower'} yield, a {strength_p} rank link "
    f"(Spearman = {rho_pest:.2f}). That is **not** cause-and-effect: countries that farm intensively use more pesticide and also tend to have "
    f"other advantages (irrigation, seed, fertiliser). Inside a single country the link between a heavier-than-usual pesticide year and "
    f"yield is r = {within_p:.2f}, so most of the pattern is between countries rather than a lever a ministry can pull.")

**Interpretation of chart 4:** TODO (does more pesticide go with more yield? Is that cause-and-effect?)

**Correlation table.**

In [ ]:
# TODO: show correlations of yield with rain_mm, pesticides_t, temp_c, Year (HINT: rice[[...]].corr()["yield_hg_ha"]) and store in RESULTS["corr_with_yield"].
# YOUR CODE HERE
corr = rice[["yield_hg_ha", "rain_mm", "pesticides_t", "temp_c", "Year"]].corr()["yield_hg_ha"].drop("yield_hg_ha")   # correlation of yield with each input
display(corr.to_frame("correlation with yield"))                        # show the table
RESULTS["corr_with_yield"] = {k: float(v) for k, v in corr.items()}     # store headline numbers

In [ ]:
top_feat = corr.abs().idxmax()                                          # strongest single correlation
say("### Written answer for Section 5\n\n"
    f"> **This means the ministry should** not expect any single weather or input figure to forecast yield: the strongest simple correlation "
    f"is {top_feat} (r = {corr[top_feat]:.2f}) and most of the pattern lies between countries. It should plan by country and farming system, "
    f"and read pesticide use as a marker of intensive farming rather than a lever to pull.")

### ✍️ Your written answer for Section 5 (double-click to edit)
> **This means the ministry should…** *(TODO: finish this sentence in plain English, using the numbers you found above. Maximum 3 sentences.)*

---
# SECTION 6 · Part 4 – Regression (a): choose inputs, encode, split
**Assigned to:** Member 6 (write your name here: ________)  |  **Marks area:** Regression (20 marks) – milestone 5 (shared with Member 7)

**Receives:** `rice`, `RANDOM_STATE`, `RESULTS`
**Hands over:** `X`, `y`, `X_train`, `X_test`, `y_train`, `y_test`, `baseline_rmse`, `num_features`

**Goal:** build the input table X and target y and split correctly. **Scoring on training data later costs −7**, so get the split right.

**Step 6.1 – Choose inputs.**

In [ ]:
# TODO: set TARGET = "yield_hg_ha" and num_features = ["Year", "pesticides_t", "temp_c"].
# THINK: rain_mm is ONE value per country (Member 3 checked). Why would including both rain_mm and a country label confuse a linear model?
#        Write your answer as a comment here: ...
# TODO: one-hot encode the country column and combine with the numeric features.
# HINT: pd.get_dummies(rice[["Area"] + num_features], columns=["Area"], drop_first=True, dtype=float)
# TODO: y = rice[TARGET]; print X.shape
# YOUR CODE HERE
TARGET = "yield_hg_ha"                                                  # what we predict
num_features = ["Year", "pesticides_t", "temp_c"]                       # numeric inputs
# THINK answer: rain_mm has ONE value per country, so it is just a weighted sum of the country labels.
#   Putting both in a linear model gives perfectly collinear inputs: the split of credit between them is arbitrary and the coefficients mean nothing.
X = pd.get_dummies(rice[["Area"] + num_features], columns=["Area"], drop_first=True, dtype=float)   # one-hot countries + numbers
y = rice[TARGET]                                                        # target column
print("X shape:", X.shape)                                              # rows x inputs

**Step 6.2 – Split.**

In [ ]:
# TODO: split into 80% train / 20% test. HINT: train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
# TODO: print the sizes of the train and test sets.
# YOUR CODE HERE
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)   # 80/20 split
print("Train rows:", len(X_train), "| Test rows:", len(X_test))         # sizes

**Step 6.3 – Baseline.**

In [ ]:
# TODO: compute the error of a 'no skill' model that always predicts the TRAINING mean yield, measured on the TEST set (RMSE).
# HINT: np.sqrt(mean_squared_error(y_test, np.full(len(y_test), y_train.mean())))  -> store in baseline_rmse and RESULTS["baseline_rmse"]
# YOUR CODE HERE
baseline_rmse = np.sqrt(mean_squared_error(y_test, np.full(len(y_test), y_train.mean())))   # 'always predict the training mean'
print(f"Baseline RMSE on the test set: {baseline_rmse:,.0f} hg/ha")     # print it
RESULTS["baseline_rmse"] = float(baseline_rmse)                         # store it

**Step 6.4 – Explain (3 sentences):** why do we split BEFORE fitting anything, and why must the final scores come from the test set?

> TODO

---
# SECTION 7 · Part 4 – Regression (b): two models, honest scores, coefficients in real units
**Assigned to:** Member 7 (write your name here: ________)  |  **Marks area:** Regression (20 marks) – milestone 5

**Receives:** `X_train, X_test, y_train, y_test, baseline_rmse, rice`
**Hands over:** `lin`, `rf` (fitted models), `reg_table`, `X`, `y` used again by Member 11

**Goal:** compare two models on the TEST set and read what they mean.

**Step 7.1 – Fit two models on TRAINING data only.**

In [ ]:
# TODO: fit LinearRegression() and RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1).
# HINT: model.fit(X_train, y_train)
# ---- solution ----
lin = LinearRegression().fit(X_train, y_train)                          # linear model on TRAINING data only
rf = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train)   # forest on TRAINING data only

**Step 7.2 – Score both models on train AND test.**

In [ ]:
# TODO: for each model compute RMSE (np.sqrt(mean_squared_error(...))) and R2 (r2_score(...)) on BOTH sets; collect into a DataFrame `reg_table`.
# WARNING: R2 above 0.95 on the TEST set means something is wrong (training data scored? duplicates leaking?).
# TODO: compare to baseline_rmse.  Store results in RESULTS["reg"].
# ---- solution ----
def rmse_r2(model, Xa, ya):
    """Return (RMSE, R2) of a fitted model on the given data."""
    pred = model.predict(Xa)                                            # predictions
    return float(np.sqrt(mean_squared_error(ya, pred))), float(r2_score(ya, pred))   # error and R2

rows = []                                                               # one row per model
for name, model in [("Linear Regression", lin), ("Random Forest", rf)]:
    tr_rmse, tr_r2 = rmse_r2(model, X_train, y_train)                   # score on TRAIN
    te_rmse, te_r2 = rmse_r2(model, X_test, y_test)                     # score on TEST (the one that counts)
    rows.append({"Model": name, "Train RMSE": tr_rmse, "Test RMSE": te_rmse, "Train R2": tr_r2, "Test R2": te_r2})
reg_table = pd.DataFrame(rows).set_index("Model")                       # collect into a table
display(reg_table)
print(f"Baseline (always the mean) test RMSE: {baseline_rmse:,.0f} hg/ha")
for name in reg_table.index:                                            # compare with baseline
    gain = 1 - reg_table.loc[name, "Test RMSE"] / baseline_rmse         # % error removed
    print(f"{name}: test RMSE is {gain:.0%} lower than the baseline")
if (reg_table["Test R2"] > 0.95).any():                                 # warning from the brief
    print("NOTE: test R2 above 0.95 - re-check the split. Here country labels carry a lot of information, so a high score can be legitimate, "
          "but the forecasting test in Step 7.7 is the honest check.")
RESULTS["reg"] = reg_table.to_dict(orient="index")                      # store headline numbers

**Step 7.3 – Say it in plain English.**

In [ ]:
# TODO: print one sentence per model such as: "typically off by about ___ hg/ha (__% of the average yield) and explains __% of the variation".
# TODO: print Train R2 minus Test R2 for each model. Which one over-fits more?
# ---- solution ----
mean_test = float(y_test.mean())                                        # average test yield
for name in reg_table.index:                                            # one sentence per model
    r_ = reg_table.loc[name]
    print(f"{name}: typically off by about {r_['Test RMSE']:,.0f} hg/ha ({r_['Test RMSE']/mean_test:.0%} of the average yield) "
          f"and explains {r_['Test R2']:.0%} of the variation in yield.")
gaps = (reg_table["Train R2"] - reg_table["Test R2"])                   # over-fitting gap
print("\nTrain R2 minus Test R2:", gaps.round(3).to_dict())
print(f"{gaps.idxmax()} over-fits more (bigger gap between train and test).")

**Step 7.4 – Coefficients in real units (linear model).**

In [ ]:
# TODO: print the Linear Regression coefficients for Year, pesticides_t and temp_c as sentences, e.g. "each extra year: +__ hg/ha".
# HINT: pd.Series(lin.coef_, index=X.columns).  Scale to a meaningful size (e.g. per +10,000 tonnes of pesticide).
# TODO (stretch): fit a second linear model WITHOUT country labels but WITH rain_mm and report the rainfall coefficient per +100 mm.
#       Why can't you put rain_mm and the country labels in the same linear model?
# ---- solution ----
coef = pd.Series(lin.coef_, index=X.columns)                            # linear coefficients
print(f"Each extra year (other inputs fixed): {coef['Year']:+,.0f} hg/ha")
print(f"Each extra 10,000 tonnes of pesticide: {coef['pesticides_t'] * 10000:+,.0f} hg/ha")
print(f"Each +1 degree C: {coef['temp_c']:+,.0f} hg/ha")
# Stretch: replace country labels with rain_mm
X2 = rice[["Year", "pesticides_t", "temp_c", "rain_mm"]]                # no country labels, but with rain
lin2 = LinearRegression().fit(X2.loc[X_train.index], y_train)           # same training rows
r2_2 = r2_score(y_test, lin2.predict(X2.loc[X_test.index]))             # same test rows
c2 = pd.Series(lin2.coef_, index=X2.columns)                            # coefficients
print(f"\nNo-country model: each +100 mm of rain: {c2['rain_mm'] * 100:+,.0f} hg/ha (test R2 {r2_2:.2f}, versus {reg_table.loc['Linear Regression', 'Test R2']:.2f} with countries)")
print("Why not both? rain_mm is constant inside a country, so it duplicates the country labels (perfect collinearity);"
      f" in this file rain takes at most {rain_per_country_max} value(s) per country.")

**Step 7.5 – What does the forest rely on?**

In [ ]:
# TODO: get rf.feature_importances_, add up the importance of all 'Area_' columns into one 'Country' bar and chart it with Year, pesticides, temperature.
# ---- solution ----
imp = pd.Series(rf.feature_importances_, index=X.columns)               # importance of every input
country_imp = imp[imp.index.str.startswith("Area_")].sum()              # add up all country columns
grouped = pd.concat([pd.Series({"Country": country_imp}), imp[num_features]]).sort_values()   # four bars
fig, ax = plt.subplots(figsize=(7, 3.5))                                # new figure
ax.barh(grouped.index, grouped.values, color="#4C9F70")                 # horizontal bars
ax.set_title("What the random forest relies on")                        # title
ax.set_xlabel("Share of total importance")                              # x label
ax.set_ylabel("Input")                                                  # y label
save_fig(fig, "reg_feature_importance")                                 # save image
plt.show()                                                              # display
print(grouped.sort_values(ascending=False).round(3).to_dict())

**Step 7.6 – Actual vs predicted chart (test set).**

In [ ]:
# TODO: scatter of y_test (x) vs rf.predict(X_test) (y) with a dashed y = x line. Title, labels, save_fig(fig, "reg_actual_vs_pred").
# ---- solution ----
pred_rf = rf.predict(X_test)                                            # forest predictions on TEST
fig, ax = plt.subplots(figsize=(6, 6))                                  # new figure
ax.scatter(y_test, pred_rf, alpha=0.4, s=18, color="#3F6FB5")           # actual vs predicted
lims = [min(y_test.min(), pred_rf.min()), max(y_test.max(), pred_rf.max())]   # axis limits
ax.plot(lims, lims, "--", color="black", label="Perfect prediction (y = x)")  # y = x line
ax.set_title("Random forest: actual vs predicted yield (test set)")     # title
ax.set_xlabel("Actual yield (hg/ha)")                                   # x label
ax.set_ylabel("Predicted yield (hg/ha)")                                # y label
ax.legend()                                                             # legend
save_fig(fig, "reg_actual_vs_pred")                                     # save image
plt.show()                                                              # display

**Step 7.7 – A harder, more honest test: forecast the future.**

In [ ]:
# TODO: train on years <= 2008 and test on years 2009-2013 (use a boolean mask on rice["Year"]); report RMSE and R2 for both models.
# Why is this score lower than the random-split score, and which one should the ministry trust for planning imports?
# ---- solution ----
tr_mask = (rice["Year"] <= 2008).to_numpy()                             # training years
te_mask = (rice["Year"] >= 2009).to_numpy()                             # future years
assert te_mask.sum() > 0 and tr_mask.sum() > 0, "No rows in one of the periods"
Xf_tr, Xf_te, yf_tr, yf_te = X[tr_mask], X[te_mask], y[tr_mask], y[te_mask]   # time-based split
lin_f = LinearRegression().fit(Xf_tr, yf_tr)                            # linear model on the past
rf_f = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1).fit(Xf_tr, yf_tr)   # forest on the past
base_f = float(np.sqrt(mean_squared_error(yf_te, np.full(len(yf_te), yf_tr.mean()))))   # baseline for the future
rows = []
for name, model in [("Linear Regression", lin_f), ("Random Forest", rf_f)]:
    rm, r2v = rmse_r2(model, Xf_te, yf_te)                              # score on the future
    rows.append({"Model": name, "Forecast RMSE": rm, "Forecast R2": r2v,
                 "Random-split RMSE": reg_table.loc[name, "Test RMSE"], "Random-split R2": reg_table.loc[name, "Test R2"]})
fc_table = pd.DataFrame(rows).set_index("Model")                        # compare both tests
display(fc_table)
print(f"Baseline RMSE when forecasting 2009-2013: {base_f:,.0f} hg/ha")
print("The forecast score is usually worse because a random split lets the model peek at neighbouring years of the same country "
      "(interpolation), whereas forecasting must extrapolate into years it never saw; trees cannot extend a trend beyond the training range.")
print("The ministry should trust the forecast score for planning imports: that is the situation it will actually face.")
RESULTS["reg_forecast"] = fc_table.to_dict(orient="index")              # store headline numbers

### ✍️ Your written answer for Section 7 (double-click to edit)
> This means the ministry should use the Random Forest for import and storage planning but quote its forward-looking error, about +/-34,476 hg/ha (17% of average yield, R2 0.89), rather than the friendlier 23,910 from a random split. It should hold buffer stocks sized to that error and read the coefficients as patterns between countries, not as promises about what an extra tonne of pesticide will do.



---
# SECTION 8 · Part 5 – Classification (a): define 'bad year', avoid leakage, split
**Assigned to:** Member 8 (write your name here: ________)  |  **Marks area:** Classification (20 marks) – milestone 6 (shared with Member 9)

**Receives:** `rice`, `RANDOM_STATE`
**Hands over:** `idx_train`, `idx_test`, `Xc_train`, `Xc_test`, `yc_train`, `yc_test`, `clf_features`

**Goal:** create the yes/no target and choose inputs so the model cannot read the answer. **A yield column among the inputs costs −10.**

**Step 8.1 – Split first.**

In [ ]:
# TODO: split the row INDEX of rice 80/20: idx_train, idx_test = train_test_split(rice.index, test_size=0.2, random_state=RANDOM_STATE)
# ---- solution ----
idx_train, idx_test = train_test_split(rice.index, test_size=0.2, random_state=RANDOM_STATE)   # split the ROW INDEX first
print("Train rows:", len(idx_train), "| Test rows:", len(idx_test))

**Step 8.2 – Define a bad year.**

In [ ]:
# TODO: decide what 'bad' means. Suggested rule: yield more than 15% below that COUNTRY's own typical (median) yield.
# WHY per country? (write your reason as a comment) ...
# TODO: compute each country's median yield using TRAINING rows only: rice.loc[idx_train].groupby("Area")["yield_hg_ha"].median()
# TODO: create rice["typical_yield"] (map the medians onto every row) and rice["bad_year"] = (yield < 0.85 * typical).astype(int)
# TODO: check for countries with no training rows (typical_yield is NaN) and exclude them. Print the % of bad years in train and test.
# ---- solution ----
# WHY per country? Typical yields differ many-fold between countries. One national cut-off would call every normal year in a low-yield
# country "bad" and never flag a disaster in a high-yield one. A shortfall against the country's OWN norm is what hurts farmers.
train_median = rice.loc[idx_train].groupby("Area")["yield_hg_ha"].median()          # typical yield from TRAINING rows only
rice["typical_yield"] = rice["Area"].map(train_median)                                # map onto every row (NaN if no training rows)
rice["bad_year"] = np.where(rice["typical_yield"].isna(), np.nan,
                            (rice["yield_hg_ha"] < 0.85 * rice["typical_yield"]).astype(float))   # 1 = more than 15% below own norm
no_train = rice["typical_yield"].isna()                                               # countries with no training rows
print("Rows in countries with no training rows (excluded):", int(no_train.sum()))
idx_train = idx_train[~no_train.loc[idx_train].to_numpy()]                            # drop them from train
idx_test = idx_test[~no_train.loc[idx_test].to_numpy()]                               # drop them from test
print(f"Bad years: {rice.loc[idx_train, 'bad_year'].mean():.1%} of train rows, {rice.loc[idx_test, 'bad_year'].mean():.1%} of test rows")

**Step 8.3 – Choose inputs WITHOUT leakage.**

In [ ]:
# TODO: set clf_features = ["Year", "rain_mm", "pesticides_t", "temp_c"].
# TODO: write an assert that fails if any input name contains 'yield', or equals 'typical_yield' or 'bad_year'.
# THINK: why is 'typical_yield' also leakage even though it does not say 'bad'? (comment your answer)
# TODO: build Xc_train, Xc_test, yc_train, yc_test from the indices.
# ---- solution ----
clf_features = ["Year", "rain_mm", "pesticides_t", "temp_c"]                          # inputs that cannot reveal the answer
assert not any(("yield" in f.lower()) or f in ("typical_yield", "bad_year") for f in clf_features), "LEAKAGE: a yield-based column is among the inputs"
# THINK answer: typical_yield is computed from the yield column and the label compares yield with it, so it hands the model
#   half of the label's own formula. Any feature derived from yield is a yield column in disguise.
Xc_train, Xc_test = rice.loc[idx_train, clf_features], rice.loc[idx_test, clf_features]   # inputs
yc_train, yc_test = rice.loc[idx_train, "bad_year"].astype(int), rice.loc[idx_test, "bad_year"].astype(int)   # targets
print("Inputs:", clf_features, "| train", Xc_train.shape, "| test", Xc_test.shape)

**Step 8.4 – Explain (3 sentences):** why do we define 'bad' relative to each country rather than with one national cut-off?

> TODO

---
# SECTION 9 · Part 5 – Classification (b): models, accuracy / precision / recall, confusion matrix
**Assigned to:** Member 9 (write your name here: ________)  |  **Marks area:** Classification (20 marks) – milestone 6

**Receives:** `Xc_train, Xc_test, yc_train, yc_test`
**Hands over:** `clf_table`, `preds`, `best_name`

**Goal:** build a warning model and explain its mistakes in plain English.

**Step 9.1 – Models.**

In [ ]:
# TODO: create three models in a dict: DummyClassifier(strategy="most_frequent") as a baseline; a Pipeline of StandardScaler + LogisticRegression(max_iter=1000, class_weight="balanced");
#       RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE).
# HINT: ConvergenceWarning means scale first and raise max_iter.
# TODO: fit each on the TRAINING data and predict the TEST data.
# YOUR CODE HERE
models = {
    "Baseline (always normal)": DummyClassifier(strategy="most_frequent"),                                    # no-skill baseline
    "Logistic Regression": Pipeline([("scale", StandardScaler()),                                              # scale first
                                     ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))]),  # balanced classes
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
}
preds = {}                                                                # test predictions per model
for name, model in models.items():
    model.fit(Xc_train, yc_train)                                         # fit on TRAINING data
    preds[name] = model.predict(Xc_test)                                  # predict TEST data


**Step 9.2 – Metrics table.**

In [ ]:
# TODO: for each model compute accuracy, precision, recall (and F1) on the TEST set; collect into a DataFrame `clf_table`.
# HINT: precision_score(y_test, pred, zero_division=0)
# WARNING: accuracy of 1.00 = leakage. Go back to Section 8.
# YOUR CODE HERE
rows = []
for name, p in preds.items():                                             # metrics on the TEST set
    rows.append({"Model": name,
                 "Accuracy": accuracy_score(yc_test, p),
                 "Precision": precision_score(yc_test, p, zero_division=0),
                 "Recall": recall_score(yc_test, p, zero_division=0),
                 "F1": f1_score(yc_test, p, zero_division=0)})
clf_table = pd.DataFrame(rows).set_index("Model")                         # collect into a table
display(clf_table)
if (clf_table["Accuracy"] >= 0.999).any():                                # leakage alarm from the brief
    print("WARNING: accuracy of 1.00 means leakage - go back to Section 8.")
RESULTS["clf"] = clf_table.to_dict(orient="index")                        # store headline numbers

**Step 9.3 – Confusion matrices.**

In [ ]:
# TODO: draw a confusion matrix for each real model. HINT: ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=[...]).plot(ax=ax)
# TODO: save_fig(fig, "clf_confusion")
# YOUR CODE HERE
real_models = [n for n in preds if not n.startswith("Baseline")]          # the two real models
fig, axes = plt.subplots(1, len(real_models), figsize=(10, 4))            # one panel each
for ax, name in zip(axes, real_models):
    ConfusionMatrixDisplay(confusion_matrix(yc_test, preds[name]), display_labels=["Normal", "Bad year"]).plot(ax=ax, colorbar=False)   # draw the matrix
    ax.grid(False)                                                        # remove grid lines drawn over the matrix
    ax.set_title(name)                                                    # panel title
    ax.set_xlabel("Predicted")                                            # x label
    ax.set_ylabel("Actual")                                               # y label
fig.suptitle("Confusion matrices on the test set")                        # overall title
save_fig(fig, "clf_confusion")                                            # save image
plt.show()                                                                # display

**Step 9.4 – Explain the mistakes in plain English.**

In [ ]:
# TODO: pick the best real model (justify: which matters more to the ministry, missing a bad year or a false alarm?).
# TODO: unpack tn, fp, fn, tp = confusion_matrix(...).ravel() and print one sentence for EACH of the four numbers
#       (e.g. "30 bad years looked normal to the model -> farmers get no early support").
# TODO: compare accuracy with the 'always normal' baseline. Why is accuracy misleading when bad years are rare?
# YOUR CODE HERE
f2 = {n: fbeta_score(yc_test, preds[n], beta=2, zero_division=0) for n in real_models}   # F2 weights recall more than precision
best_name = max(f2, key=f2.get)                                           # best real model
print(f"Best model: {best_name} (F2 = {f2[best_name]:.2f}). Missing a bad year (no early support reaches farmers) costs the ministry far more "
      f"than a false alarm (an unnecessary field check), so recall is weighted above precision.")
tn, fp, fn, tp = confusion_matrix(yc_test, preds[best_name]).ravel()      # the four counts
print(f"- {tn:,} normal years were correctly left alone -> no wasted effort")
print(f"- {fp:,} normal years were flagged as bad -> false alarms: support or checks sent that were not needed")
print(f"- {fn:,} bad years looked normal to the model -> farmers in those years get no early warning")
print(f"- {tp:,} bad years were caught -> early support can be sent in time")
base_acc = clf_table.loc["Baseline (always normal)", "Accuracy"]           # accuracy of the no-skill model
print(f"\nAn 'always normal' model scores {base_acc:.1%} accuracy while catching 0 bad years; {best_name} scores {clf_table.loc[best_name, 'Accuracy']:.1%}. "
      f"Accuracy misleads when bad years are rare because the common class dominates the score, so precision and recall are the honest measures.")

In [ ]:
rec_ = clf_table.loc[best_name, "Recall"]                                 # share of bad years caught
prec_ = clf_table.loc[best_name, "Precision"]                             # share of alerts that were right
say("### Written answer for Section 9\n\n"
    f"> **This means the ministry should** use the {best_name} as a screening alarm that catches {rec_:.0%} of bad years, while accepting that only "
    f"{prec_:.0%} of its alerts are real and {fn:,} bad years ({fn/max(fn+tp, 1):.0%}) still slip through. Every alert should trigger a field check rather than automatic payouts, "
    f"and the ministry should judge the system by recall and precision, never by accuracy alone.")

### ✍️ Your written answer for Section 9 (double-click to edit)
> **This means the ministry should…** *(TODO: finish this sentence in plain English, using the numbers you found above. Maximum 3 sentences.)*

---
# SECTION 10 · Part 6 – Clustering: what kinds of growing years do we have?
**Assigned to:** Member 10 (write your name here: ________)  |  **Marks area:** Clustering (10 marks) – milestone 7

**Receives:** `rice`, `RANDOM_STATE`, `RESULTS`
**Hands over:** `rice['cluster']`, `rice['growing_type']`, `summary` table

**Goal:** scale, choose k with evidence, fit K-Means and give every group a plain-English name.

**Step 10.1 – Scale first.**

In [ ]:
# TODO: choose cluster columns (e.g. yield_hg_ha, rain_mm, pesticides_t, temp_c) and scale them with StandardScaler().fit_transform(...)
# WHY? (comment). If every point lands in one cluster you forgot this step.
# YOUR CODE HERE
cluster_cols = ["yield_hg_ha", "rain_mm", "pesticides_t", "temp_c"]        # what defines a growing year
cdata = rice[cluster_cols].copy()                                          # copy so rice is untouched
cdata["pesticides_t"] = np.log1p(cdata["pesticides_t"])                    # log: pesticide use is extremely skewed
# WHY scale? K-Means uses distances; pesticide tonnes (thousands) would swamp temperature (tens) and one input would decide everything.
Z = StandardScaler().fit_transform(cdata)                                  # mean 0, std 1 per column
print("Scaled means:", Z.mean(axis=0).round(2), "| stds:", Z.std(axis=0).round(2))

**Step 10.2 – Choose k with evidence.**

In [ ]:
# TODO: for k in 2..8 fit KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE); record inertia_ and silhouette_score(Z, labels_).
# TODO: draw an elbow plot and a silhouette plot, save_fig(fig, "cluster_choose_k").
# TODO: also look at the cluster SIZES for k=3,4,5. A tiny cluster (<3% of rows) is usually not a useful 'type'.
# TODO: write the justification of your k in a markdown cell below.
# YOUR CODE HERE
ks = range(2, 9)                                                           # k values to try
inertias, sils, sizes = {}, {}, {}
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(Z)  # fit k clusters
    inertias[k] = km.inertia_                                              # within-cluster spread
    sils[k] = silhouette_score(Z, km.labels_)                              # separation quality
    sizes[k] = pd.Series(km.labels_).value_counts(normalize=True).sort_index()   # cluster shares
fig, axes = plt.subplots(1, 2, figsize=(11, 4))                            # elbow + silhouette
axes[0].plot(list(ks), list(inertias.values()), marker="o")                # elbow
axes[0].set_title("Elbow plot")
axes[0].set_xlabel("Number of clusters k")
axes[0].set_ylabel("Inertia (within-cluster spread)")
axes[1].plot(list(ks), list(sils.values()), marker="o", color="#D1603D")   # silhouette
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("Number of clusters k")
axes[1].set_ylabel("Silhouette score")
save_fig(fig, "cluster_choose_k")                                          # save image
plt.show()
for k in (3, 4, 5):                                                        # cluster sizes
    print(f"k={k}: sizes (% of rows) =", (sizes[k] * 100).round(1).tolist(), "| silhouette", round(sils[k], 3))
candidates = [k for k in range(3, 7) if sizes[k].min() >= 0.03]            # no tiny clusters, and more than 'high vs low'
K = max(candidates, key=sils.get) if candidates else 3                     # best silhouette among sensible k
silhouette = float(sils[K])                                                # silhouette of the chosen k
RESULTS["clustering"] = {"k": int(K), "silhouette": silhouette, "smallest_cluster_share": float(sizes[K].min())}
print("Chosen K =", K)

**Step 10.3 – Fit, profile, name.**

In [ ]:
# TODO: fit the final KMeans and add rice["cluster"].
# TODO: profile each cluster: mean of each column, number of rows, number of countries. HINT: rice.groupby("cluster")[cols].mean()
# TODO: give EVERY cluster a plain-English name that a minister understands (e.g. 'Cool, high-input, high-yield'), stored in rice["growing_type"].
# HINT: print example countries per cluster: rice[rice.cluster==c]["Area"].value_counts().head(5)
# YOUR CODE HERE
kmeans = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(Z)  # final model
rice["cluster"] = kmeans.labels_                                           # add the label to every row
prof = rice.groupby("cluster")[cluster_cols].mean()                        # average of each column per cluster
prof["n_rows"] = rice.groupby("cluster").size()                            # rows per cluster
prof["n_countries"] = rice.groupby("cluster")["Area"].nunique()            # countries per cluster
zc = pd.DataFrame(Z, columns=cluster_cols).groupby(rice["cluster"].to_numpy()).mean()   # cluster means in standard units

def word(z, low, mid, high):
    """Turn a standardised average into a plain-English word."""
    return high if z > 0.5 else low if z < -0.5 else mid                   # +/- half a standard deviation

names = {}
for c in zc.index:                                                         # name every cluster
    names[c] = ", ".join([word(zc.loc[c, "temp_c"], "Cool", "Mild", "Hot"),
                          word(zc.loc[c, "rain_mm"], "dry", "moderate-rain", "wet"),
                          word(zc.loc[c, "pesticides_t"], "low-input", "mid-input", "high-input"),
                          word(zc.loc[c, "yield_hg_ha"], "low-yield", "mid-yield", "high-yield")])
seen = {}
for c in list(names):                                                      # make names unique
    seen[names[c]] = seen.get(names[c], 0) + 1
    if seen[names[c]] > 1:
        names[c] += f" ({seen[names[c]]})"
rice["growing_type"] = rice["cluster"].map(names)                          # plain-English name on every row
summary = prof.copy()
summary["examples"] = [", ".join(rice[rice.cluster == c]["Area"].value_counts().head(3).index) for c in summary.index]   # example countries
summary.index = [names[c] for c in summary.index]                          # row labels = names
display(summary)
RESULTS["clustering"]["types"] = {n: {"n_rows": int(r["n_rows"]), "mean_yield": float(r["yield_hg_ha"])} for n, r in summary.iterrows()}

**Step 10.4 – Visualise and interpret.**

In [ ]:
# TODO: scatter plot (e.g. temp_c vs yield) coloured by growing_type, with title, labels, legend. save_fig(fig, "cluster_scatter")
# TODO: check how many countries stay in ONE cluster in every year: rice.groupby("Area")["cluster"].nunique() == 1
#       What does that say about what drives the groups - weather from year to year, or the farming system of each country?
# YOUR CODE HERE
fig, ax = plt.subplots(figsize=(9, 6))                                     # new figure
for t in summary.index:                                                    # one colour per type
    sub = rice[rice["growing_type"] == t]
    ax.scatter(sub["temp_c"], sub["yield_hg_ha"], alpha=0.5, s=18, label=t)
ax.set_title(f"{CROP}: growing-year types")                                # title
ax.set_xlabel("Average temperature (degrees C)")                           # x label with units
ax.set_ylabel("Yield (hg/ha)")                                             # y label
ax.legend(fontsize=8)                                                      # legend
save_fig(fig, "cluster_scatter")                                           # save image
plt.show()
stable = (rice.groupby("Area")["cluster"].nunique() == 1)                  # countries that never change group
stable_share = float(stable.mean())                                        # share of such countries
print(f"{stable.sum()} of {len(stable)} countries ({stable_share:.0%}) stay in ONE cluster in every year.")
print("What drives the groups? " + ("Mostly each country's farming system (country-level inputs and yield barely change group over time)."
      if stable_share >= 0.6 else "A mix of country level and year-to-year weather, since many countries move between groups."))

### ✍️ Your written answer for Section 10 (double-click to edit)
> **This means the ministry should…** *(TODO: finish this sentence in plain English, using the numbers you found above. Maximum 3 sentences.)*

In [ ]:
big = summary["n_rows"].idxmax()                                           # largest type
low = summary["yield_hg_ha"].idxmin()                                      # lowest-yield type
tail_txt = (f"Because {stable_share:.0%} of countries never leave their group, the types mainly describe each country's farming system, so moving a country up takes investment, not just a good season."
            if stable_share >= 0.6 else
            f"Because only {stable_share:.0%} of countries stay in one group all the time, countries do move between types from year to year, so the ministry should re-classify each season.")
say("### Written answer for Section 10\n\n"
    f"> **This means the ministry should** plan by growing type, not by one national average: the largest group, '{big}', holds "
    f"{summary.loc[big, 'n_rows']/summary['n_rows'].sum():.0%} of country-years, while '{low}' averages only {summary.loc[low, 'yield_hg_ha']:,.0f} hg/ha "
    f"across {int(summary.loc[low, 'n_countries'])} countries and should get support first. {tail_txt}")

In [ ]:
say(f"**Why k = {K}?** The elbow plot flattens gradually rather than showing one sharp bend, so it is only a rough guide. Among k = 3 to 6 I kept those "
    f"whose smallest cluster holds at least 3% of rows and picked the best silhouette, which is k = {K} (silhouette {silhouette:.2f}; "
    f"smallest group {sizes[K].min():.0%} of rows). k = 2 was excluded even if it scores higher, because it only splits 'high' from 'low' "
    f"and the ministry asked for kinds of growing year. A silhouette of {silhouette:.2f} means the groups overlap somewhat, so treat them as useful "
    f"labels rather than sharp boundaries.")

**Why this k? (edit):** TODO – use the elbow, the silhouette scores AND the cluster sizes.

---
# SECTION 11 · Part 7 – Tune with Grid Search, final summary, and hand-in check
**Assigned to:** Member 11 (write your name here: ________)  |  **Marks area:** Tuning (5 marks) – milestone 7, plus final integration

**Receives:** `X_train, X_test, y_train, y_test`, `rf`, `RESULTS`, `clf_table`, `best_name`, `K`, `silhouette`
**Hands over:** `results/summary.json`

**Goal:** run Grid Search on your best regression model, report before vs after *honestly*, and assemble the final scoreboard.

**Step 11.1 – Grid Search.**

In [ ]:
# TODO: define a small param_grid for the Random Forest, e.g. {"n_estimators": [200, 400], "max_depth": [None, 20], "min_samples_leaf": [1, 2, 4]}
# TODO: run GridSearchCV(RandomForestRegressor(random_state=RANDOM_STATE), param_grid, cv=5, scoring="neg_root_mean_squared_error").fit(X_train, y_train)
# Only the TRAINING data may be used during tuning. Print grid.best_params_ and the cross-validated RMSE (remember the minus sign!).
# YOUR CODE HERE
param_grid = {"n_estimators": [200, 400], "max_depth": [None, 20], "min_samples_leaf": [1, 2, 4]}   # 12 combinations
grid = GridSearchCV(RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1), param_grid, cv=5,
                    scoring="neg_root_mean_squared_error").fit(X_train, y_train)   # TRAINING data only
print("Best parameters:", grid.best_params_)
print(f"Cross-validated RMSE of the best setting: {-grid.best_score_:,.0f} hg/ha (the score is negative, so we flip the sign)")

**Step 11.2 – Before vs after, honestly.**

In [ ]:
# TODO: score the original `rf` and the tuned grid.best_estimator_ on the TEST set (RMSE and R2) and show them side by side.
# TODO: calculate the % change. If tuning did NOT help, say so - that is a valid, marked result.
# YOUR CODE HERE
rmse_before, r2_before = rmse_r2(rf, X_test, y_test)                       # original forest on TEST
tuned = grid.best_estimator_                                               # tuned forest
rmse_after, r2_after = rmse_r2(tuned, X_test, y_test)                      # tuned forest on TEST
cmp_ = pd.DataFrame({"RMSE": [rmse_before, rmse_after], "R2": [r2_before, r2_after]}, index=["Before tuning", "After tuning"])
display(cmp_)
pct_change = (rmse_after - rmse_before) / rmse_before * 100                # % change in error
print(f"RMSE change: {pct_change:+.1f}% | R2 change: {r2_after - r2_before:+.4f}")
print("Tuning helped noticeably." if pct_change <= -2 else "Tuning made no meaningful difference (less than 2% change) - that is a valid, honest result.")
RESULTS["tuning"] = {"rmse_before": rmse_before, "rmse_after": rmse_after, "r2_before": r2_before, "r2_after": r2_after, "best_params": grid.best_params_}

**Step 11.3 – Final scoreboard.**

In [ ]:
# TODO: build a 3-row table: Question | Method | Headline test result, for regression, classification and clustering.
# TODO: save RESULTS to results/summary.json with json.dump(..., default=float)
# YOUR CODE HERE
tn, fp, fn, tp = confusion_matrix(yc_test, preds[best_name]).ravel()
rec_ = clf_table.loc[best_name, "Recall"]
prec_ = clf_table.loc[best_name, "Precision"]

say("### Final recommendations to the ministry\n\n"
    f"1. *Forecasting:* This means the ministry should plan imports from the {best_fc} and expect errors of about +/-{f_rmse:,.0f} hg/ha "
    f"({f_rmse/float(yf_te.mean()):.0%} of average yield) in years it has not seen, holding buffer stocks of that size.\n\n"
    f"2. *Early warning:* This means the ministry should use the {best_name} as a screening alarm ({rec_:.0%} of bad years caught, {prec_:.0%} of alerts real) "
    f"and send a field check on every alert, tracking the {fn:,} misses.\n\n"
    f"3. *Planning by type:* This means the ministry should manage {K} growing types rather than one national average, starting with '{low}', "
    f"the lowest-yield type ({summary.loc[low, 'yield_hg_ha']:,.0f} hg/ha on average).")

**Step 11.4 – Hand-in check (tick each box by editing this cell).**

- [ ] Kernel → Restart & Run All finishes with no error
- [ ] `yield_df.csv` is in the repository folder and README explains how to run
- [ ] Every section ends with 'This means the ministry should…'
- [ ] Every member has commits
- [ ] No yield column among classification inputs; regression scored on TEST data
- [ ] AI use noted in the table below

In [ ]:
# (No code - tick the boxes above.)

### ✍️ Your written answer for Section 11 (double-click to edit)
> **This means the ministry should…** *(TODO: finish this sentence in plain English, using the numbers you found above. Maximum 3 sentences.)*

---
# Wrap-up (whole team)
### AI-assistant use log (honesty table)
| Section | Member | What you asked the AI for | What you changed or verified yourself |
|---|---|---|---|
| | | | |

### Final recommendations to the ministry (write together after all sections are finished)
1. **Forecasting:** This means the ministry should… TODO
2. **Early warning:** This means the ministry should… TODO
3. **Planning by type:** This means the ministry should… TODO